# No Redefinition for Units of Measure and Constants

In this notebook, we delve deeper into the different experiments regarding the No Redefinition task. Specifically, we outline the experimental setup for both units of measure and constants, across free-form and multiple-choice formats. We employ zero-shot, chain-of-thought, and few-shot prompting techniques. To assess model performance, we use an LLM evaluator.

In [ ]:
!pip install --upgrade boto3 
!pip install --upgrade botocore
!pip install langchain

In [ ]:
# Import required libraries
import boto3
import json
from botocore.exceptions import ClientError
from langchain.prompts import PromptTemplate

# AWS Bedrock setup 
client = boto3.client(
    service_name="bedrock-runtime",
    aws_access_key_id='your_aws_access_key_id',
    aws_secret_access_key='your_aws_secret_access_key',
    region_name="us-west-2"
)

In [ ]:
model_id = "amazon.titan-tg1-large"

template_z = """
Answer the following question:
{question}

End the response wi

th the phrase "The final answer is: " followed only by the correct result, with no additional text or commentary.
"""

template_cot = """
Answer the following question:
{question}

Let's think step by step. 

End the response with the phrase "The final answer is: " followed only by the correct result, with no additional text or commentary.
"""

template_f_units = """
Answer the following question:
{question}

Here are some examples of similar questions with their correct answers:

Question: How many hours are in 2 days?
Answer: 48

Question: How many radians are in a full circle?
Answer: 2*pi

Question: A lightbulb has a resistance of 50 ohms. What is its resistance in milliohms?
Answer: 50,000

End the response with the phrase "The final answer is: " followed only by the correct result, with no additional text or commentary.
"""

template_f_constants = """
Answer the following question:
{question}

Here are some examples of similar questions with their correct answers:

Question: What is 10*tau?
Answer: 62.83

Question: What is one percent of 50?
Answer: 0.5

Question: What is the first digit of vacuum magnetic permeability?
Answer: 1

End the response with the phrase "The final answer is: " followed only by the correct result, with no additional text or commentary.
"""

template_z_mc = """
Choose A, B, C or D to answer the question:

Question: {question}
A: {A}
B: {B}
C: {C}
D: {D}

Provide only the letter corresponding to the correct answer: "A", "B", "C", or "D".
End the response with the phrase "The final answer is: " followed by the correct letter, with no additional text or commentary.
"""

template_cot_mc = """
Choose A, B, C or D to answer the question:

Question: {question}
A: {A}
B: {B}
C: {C}
D: {D}

Let's think step by step. 

Provide only the letter corresponding to the correct answer: "A", "B", "C", or "D".
End the response with the phrase "The final answer is: " followed by the correct letter, with no additional text or commentary.
"""

template_f_mc_units = """
Choose A, B, C or D to answer the question:

Question: {question}
A: {A}
B: {B}
C: {C}
D: {D}

Here are some examples of similar questions with their correct answers:

Question: How many hours are in 2 days?
A: 10
B: 20
C: 24
D: 48
Answer: D

Question: How many radians are in a full circle?
A: pi
B: 2*pi
C: 180
D: 360
Answer: B

Question: A lightbulb has a resistance of 50 ohms. What is its resistance in milliohms?
A: 5,000 milliohms
B: 10,000 milliohms
C: 50,000 milliohms
D: 100,000 milliohms
Answer: C

Provide only the letter corresponding to the correct answer: "A", "B", "C", or "D".
End the response with the phrase "The final answer is: " followed by the correct letter, with no additional text or commentary.
"""

template_f_mc_constants = """
Choose A, B, C or D to answer the question:

Question: {question}
A: {A}
B: {B}
C: {C}
D: {D}

Here are some examples of similar questions with their correct answers:

Question: What is 10*tau?
A: 3.14
B: 62.83
C: 90
D: 9
Answer: B

Question: What is one percent of 50?
A: 0.5
B: 5
C: 10
D: 50
Answer: A

Question: What is the first digit of vacuum magnetic permeability?
A: 1
B: 2
C: 3
D: 4
Answer: A

Provide only the letter corresponding to the correct answer: "A", "B", "C", or "D".
End the response with the phrase "The final answer is: " followed by the correct letter, with no additional text or commentary.
"""

prompt_template_z = PromptTemplate(
    input_variables=["question"],
    template=template_z
)

prompt_template_cot = PromptTemplate(
    input_variables=["question"],
    template=template_cot
)

prompt_template_f_units = PromptTemplate(
    input_variables=["question"],
    template=template_f_units
)

prompt_template_f_constants = PromptTemplate(
    input_variables=["question"],
    template=template_f_constants
)

prompt_template_z_mc = PromptTemplate(
    input_variables=["question", "A", "B", "C", "D"],
    template=template_z_mc
)

prompt_template_cot_mc = PromptTemplate(
    input_variables=["question", "A", "B", "C", "D"],
    template=template_cot_mc
)

prompt_template_f_mc_units = PromptTemplate(
    input_variables=["question", "A", "B", "C", "D"],
    template=template_f_mc_units
)

prompt_template_f_mc_constants = PromptTemplate(
    input_variables=["question", "A", "B", "C", "D"],
    template=template_f_mc_constants
)

SYSTEM_PROMPT = "You are a helpful assistant."

In [ ]:
import json
import time
import random
from botocore.exceptions import ClientError


def get_model_response(input_text, retries=5, backoff_factor=2, max_delay=60):
    native_request = {
        "inputText": input_text,
        "textGenerationConfig": {
            "maxTokenCount": 512,
            "temperature": 0.5,
            "topP": 0.9
        }
    }

    request = json.dumps(native_request)

    for attempt in range(1, retries + 1):
        try:
            response = client.invoke_model(modelId=model_id, body=request)
            model_response = json.loads(response["body"].read())

            output_text = model_response["results"][0]["outputText"]
            return output_text

        except (KeyError, IndexError):
            print(f"KeyError on attempt {attempt}: {model_response}")
        except ClientError as e:
            print(f"ClientError on attempt {attempt}: {e}")
        except Exception as e:
            print(f"Error on attempt {attempt}: {e}")
        
        if attempt == retries:
            print("Max retries reached. Returning None.")
            return None

        delay = min(backoff_factor * (2 ** (attempt - 1)), max_delay) + random.uniform(0, 1)
        print(f"Retrying in {delay:.2f} seconds...")
        time.sleep(delay)

# **Experiments**

## **Units of Measure**

In [2]:
import os
import pandas as pd

is_kaggle = os.path.exists('/kaggle/input')

# If you're running this notebook in the Kaggle environment you need to manually upload the files contained in the 'input' folder
#   Upload these files to the Kaggle "Input" section before running the code.
#   Make sure the folder structure matches the expected directories:
#   - 'units-level1' folder containing 'units of measure - level1.csv' and 'units of measure - level1_mc.csv'
#   - 'units-level2' folder containing 'units of measure - level2.csv' and 'units of measure - level2_mc.csv'
#   - 'units-level3' folder containing 'units of measure - level3.csv' and 'units of measure - level3_mc.csv'

if is_kaggle:
    input_dir = '/kaggle/input'
else:
    input_dir = os.path.join(os.getcwd(), 'input')

level1_dir = os.path.join(input_dir, "units-level1")
level2_dir = os.path.join(input_dir, "units-level2")
level3_dir = os.path.join(input_dir, "units-level3")

file_level1 = ["units of measure - level1.csv", "units of measure - level1_mc.csv"]
file_level2 = ["units of measure - level2.csv", "units of measure - level2_mc.csv"]
file_level3 = ["units of measure - level3.csv", "units of measure - level3_mc.csv"]

df1_units = pd.read_csv(os.path.join(level1_dir, file_level1[0]))  
df1_units_mc = pd.read_csv(os.path.join(level1_dir, file_level1[1]))  

df2_units = pd.read_csv(os.path.join(level2_dir, file_level2[0]))  
df2_units_mc = pd.read_csv(os.path.join(level2_dir, file_level2[1])) 

df3_units = pd.read_csv(os.path.join(level3_dir, file_level3[0]))  
df3_units_mc = pd.read_csv(os.path.join(level3_dir, file_level3[1]))

In [4]:
data_level1_units = [
    {
        "X": row["X"],
        "real_value": row["real_value"],
        "question": row["question"],
        "answer_real": row["answer_real"]
    }
    for _, row in df1_units.iterrows()
]
data_level1_units[:2]

[{'X': '1 minute',
  'real_value': '60 seconds',
  'question': 'How many seconds are in 2 minutes?',
  'answer_real': '120'},
 {'X': '1 kilogram',
  'real_value': '1000 grams',
  'question': 'How many grams are in 2 kilograms?',
  'answer_real': '2000'}]

In [5]:
data_level2_units = [
    {
        "X": row["X"],
        "real_value": row["real_value"],
        "question": row["question"],
        "answer_real": row["answer_real"]
    }
    for _, row in df2_units.iterrows()
]
data_level2_units[:2]

[{'X': '1 minute',
  'real_value': '60 seconds',
  'question': 'A stopwatch runs for 3 and a half minutes. How many seconds does it count?',
  'answer_real': '210'},
 {'X': '1 kilogram',
  'real_value': '1000 grams',
  'question': 'A person weighs 72 kilograms. What is the persons weight in grams?',
  'answer_real': '72000'}]

In [6]:
data_level3_units = [
    {
        "X": row["X"],
        "real_value": row["real_value"],
        "question": row["question"],
        "answer_real": row["answer_real"]
    }
    for _, row in df3_units.iterrows()
]
data_level3_units[:2]

[{'X': '1 minute',
  'real_value': '60 seconds',
  'question': 'A marathon runner runs at a speed of 170 meters per minute. How many seconds will it take them to complete a 42-kilometer race?',
  'answer_real': '14,823.53'},
 {'X': '1 kilogram',
  'real_value': '1000 grams',
  'question': "A vehicle's engine weighs 650 kilograms. If 15% of the weight is aluminum, what is the weight of the aluminum in grams?",
  'answer_real': '97,500 grams'}]

In [7]:
data_level1_units_mc = [
    {
        "X": row["X"],
        "real_value": row["real_value"],
        "question": row["question"],
        "answer_real": row["answer_real"],
        "Y1": row["Y1"],
        "A1": row["A1"],
        "B1": row["B1"],
        "C1": row["C1"],
        "D1": row["D1"],
        "answer1": row["answer1"],
        "answer1_real": row["answer1_real"]
    }
    for _, row in df1_units_mc.iterrows()
]
data_level1_units_mc[:2]

[{'X': '1 minute',
  'real_value': '60 seconds',
  'question': 'How many seconds are in 2 minutes?',
  'answer_real': '120',
  'Y1': '100 seconds',
  'A1': '100',
  'B1': '120',
  'C1': '200',
  'D1': '220',
  'answer1': 'C',
  'answer1_real': 'B'},
 {'X': '1 kilogram',
  'real_value': '1000 grams',
  'question': 'How many grams are in 2 kilograms?',
  'answer_real': '2000',
  'Y1': '900 grams',
  'A1': '1000',
  'B1': '1500',
  'C1': '1800',
  'D1': '2000',
  'answer1': 'C',
  'answer1_real': 'D'}]

In [8]:
data_level2_units_mc = [
    {
        "X": row["X"],
        "real_value": row["real_value"],
        "question": row["question"],
        "answer_real": row["answer_real"],
        "Y1": row["Y1"],
        "A1": row["A1"],
        "B1": row["B1"],
        "C1": row["C1"],
        "D1": row["D1"],
        "answer1": row["answer1"],
        "answer1_real": row["answer1_real"]
    }
    for _, row in df2_units_mc.iterrows()
]
data_level2_units_mc[:2]

[{'X': '1 minute',
  'real_value': '60 seconds',
  'question': 'A stopwatch runs for 3 and a half minutes. How many seconds does it count?',
  'answer_real': '210',
  'Y1': '100 seconds',
  'A1': '180',
  'B1': '210',
  'C1': '300',
  'D1': '350',
  'answer1': 'D',
  'answer1_real': 'B'},
 {'X': '1 kilogram',
  'real_value': '1000 grams',
  'question': 'A person weighs 72 kilograms. What is the persons weight in grams?',
  'answer_real': '72000',
  'Y1': '900 grams',
  'A1': '47300',
  'B1': '56000',
  'C1': '64800',
  'D1': '72000',
  'answer1': 'C',
  'answer1_real': 'D'}]

In [9]:
data_level3_units_mc = [
    {
        "X": row["X"],
        "real_value": row["real_value"],
        "question": row["question"],
        "answer_real": row["answer_real"],
        "Y1": row["Y1"],
        "A1": row["A1"],
        "B1": row["B1"],
        "C1": row["C1"],
        "D1": row["D1"],
        "answer1": row["answer1"],
        "answer1_real": row["answer1_real"]
    }
    for _, row in df3_units_mc.iterrows()
]
data_level3_units_mc[:2]

[{'X': '1 minute',
  'real_value': '60 seconds',
  'question': 'A marathon runner runs at a speed of 170 meters per minute. How many seconds will it take them to complete a 42-kilometer race?',
  'answer_real': '14,823.53',
  'Y1': '100 seconds',
  'A1': '9,235.76',
  'B1': '14,823.53',
  'C1': '24,705.8824',
  'D1': '56,287.43',
  'answer1': 'C',
  'answer1_real': 'B'},
 {'X': '1 kilogram',
  'real_value': '1000 grams',
  'question': "A vehicle's engine weighs 650 kilograms. If 15% of the weight is aluminum, what is the weight of the aluminum in grams?",
  'answer_real': '97,500 grams',
  'Y1': '900 grams',
  'A1': '77,265 grams',
  'B1': '87,750 grams',
  'C1': '97,500 grams',
  'D1': '107,500 grams',
  'answer1': 'B',
  'answer1_real': 'C'}]

In [ ]:
def get_model_answer(prompt_template, question, retries=5):
    input_text = prompt_template.format(question=question)
    return get_model_response(input_text, retries=retries)

In [ ]:
def get_model_answer_mc(prompt_template, question, A, B, C, D, retries=5):
    input_text = prompt_template.format(question=question, A=A, B=B, C=C, D=D)
    return get_model_response(input_text, retries=retries)

In [ ]:
def fix_answers(llm_answers):
    llm_answers_fixed = []
    for answer in llm_answers:
        if answer is not None and 'final answer is: ' in answer:
            llm_answers_fixed.append(answer.split('final answer is: ', 1)[1])
        else:
            llm_answers_fixed.append(answer)  # Or handle it another way
    return llm_answers_fixed

### **Free Form**

In [ ]:
answers1_z1 = []
answers1_cot1 = []
answers1_f1 = []

for row in data_level1_units:
    question = row['question']
    answers1_z1.append(get_model_answer(prompt_template_z, question))
    answers1_cot1.append(get_model_answer(prompt_template_cot, question))
    answers1_f1.append(get_model_answer(prompt_template_f_units, question))

In [ ]:
answers2_z1 = []
answers2_cot1 = []
answers2_f1 = []

for row in data_level2_units:
    question = row['question']
    answers2_z1.append(get_model_answer(prompt_template_z, question))
    answers2_cot1.append(get_model_answer(prompt_template_cot, question))
    answers2_f1.append(get_model_answer(prompt_template_f_units, question))

In [ ]:
answers3_z1 = []
answers3_cot1 = []
answers3_f1 = []

for row in data_level3_units:
    question = row['question']
    answers3_z1.append(get_model_answer(prompt_template_z, question))
    answers3_cot1.append(get_model_answer(prompt_template_cot, question))
    answers3_f1.append(get_model_answer(prompt_template_f_units, question))

In [ ]:
answers1_z_fixed1 = fix_answers(answers1_z1)
answers2_z_fixed1 = fix_answers(answers2_z1)
answers3_z_fixed1 = fix_answers(answers3_z1)

answers1_cot_fixed1 = fix_answers(answers1_cot1)
answers2_cot_fixed1 = fix_answers(answers2_cot1)
answers3_cot_fixed1 = fix_answers(answers3_cot1)

answers1_f_fixed1 = fix_answers(answers1_f1)
answers2_f_fixed1 = fix_answers(answers2_f1)
answers3_f_fixed1 = fix_answers(answers3_f1)

### **Multiple Choice**

In [ ]:
answers1_z_mc1 = []
answers1_cot_mc1 = []
answers1_f_mc1 = []

for row in data_level1_units_mc:
    question = row['question']
    A1, B1, C1, D1 = row['A1'], row['B1'], row['C1'], row['D1']
    answers1_z_mc1.append(get_model_answer_mc(prompt_template_z_mc, question, A1, B1, C1, D1))
    answers1_cot_mc1.append(get_model_answer_mc(prompt_template_cot_mc, question, A1, B1, C1, D1))
    answers1_f_mc1.append(get_model_answer_mc(prompt_template_f_mc_units, question, A1, B1, C1, D1))  

In [ ]:
answers2_z_mc1 = []
answers2_cot_mc1 = []
answers2_f_mc1 = []

for row in data_level2_units_mc:
    question = row['question']
    A1, B1, C1, D1 = row['A1'], row['B1'], row['C1'], row['D1']
    answers2_z_mc1.append(get_model_answer_mc(prompt_template_z_mc, question, A1, B1, C1, D1))
    answers2_cot_mc1.append(get_model_answer_mc(prompt_template_cot_mc, question, A1, B1, C1, D1))
    answers2_f_mc1.append(get_model_answer_mc(prompt_template_f_mc_units, question, A1, B1, C1, D1)) 

In [ ]:
answers3_z_mc1 = []
answers3_cot_mc1 = []
answers3_f_mc1 = []

for row in data_level3_units_mc:
    question = row['question']
    A1, B1, C1, D1 = row['A1'], row['B1'], row['C1'], row['D1']
    answers3_z_mc1.append(get_model_answer_mc(prompt_template_z_mc, question, A1, B1, C1, D1))
    answers3_cot_mc1.append(get_model_answer_mc(prompt_template_cot_mc, question, A1, B1, C1, D1))
    answers3_f_mc1.append(get_model_answer_mc(prompt_template_f_mc_units, question, A1, B1, C1, D1)) 

In [ ]:
answers1_z_mc_fixed1 = fix_answers(answers1_z_mc1)
answers2_z_mc_fixed1 = fix_answers(answers2_z_mc1)
answers3_z_mc_fixed1 = fix_answers(answers3_z_mc1)

answers1_cot_mc_fixed1 = fix_answers(answers1_cot_mc1)
answers2_cot_mc_fixed1 = fix_answers(answers2_cot_mc1)
answers3_cot_mc_fixed1 = fix_answers(answers3_cot_mc1)

answers1_f_mc_fixed1 = fix_answers(answers1_f_mc1)
answers2_f_mc_fixed1 = fix_answers(answers2_f_mc1)
answers3_f_mc_fixed1 = fix_answers(answers3_f_mc1)

## **Constants**

In [15]:
import os
import pandas as pd

is_kaggle = os.path.exists('/kaggle/input')

# If you're running this notebook in the Kaggle environment you need to manually upload the files contained in the 'input' folder
#   Upload these files to the Kaggle "Input" section before running the code.
#   Make sure the folder structure matches the expected directories:
#   - 'constants-level1' folder containing 'constants - level1.csv' and 'constants - level1_mc.csv'
#   - 'constants-level2' folder containing 'constants - level2.csv' and 'constants - level2_mc.csv'
#   - 'constants-level3' folder containing 'constants - level3.csv' and 'constants - level3_mc.csv'

if is_kaggle:
    input_dir = '/kaggle/input'
else:
    input_dir = os.path.join(os.getcwd(), 'input')

level1_dir = os.path.join(input_dir, "constants-level1")
level2_dir = os.path.join(input_dir, "constants-level2")
level3_dir = os.path.join(input_dir, "constants-level3")

file_level1 = ["constants - level1.csv", "constants - level1_mc.csv"]
file_level2 = ["constants - level2.csv", "constants - level2_mc.csv"]
file_level3 = ["constants - level3.csv", "constants - level3_mc.csv"]

df1_constants = pd.read_csv(os.path.join(level1_dir, file_level1[0]))  
df1_constants_mc = pd.read_csv(os.path.join(level1_dir, file_level1[1]))  

df2_constants = pd.read_csv(os.path.join(level2_dir, file_level2[0]))  
df2_constants_mc = pd.read_csv(os.path.join(level2_dir, file_level2[1])) 

df3_constants = pd.read_csv(os.path.join(level3_dir, file_level3[0]))  
df3_constants_mc = pd.read_csv(os.path.join(level3_dir, file_level3[1]))

In [11]:
data_level1_constants = [
    {
        "X": row["X"],
        "real_value": row["real_value"],
        "question": row["question"],
        "answer_real": row["answer_real"]
    }
    for _, row in df1_constants.iterrows()
]
data_level1_constants[:2]

[{'X': 'pi',
  'real_value': '3.14159',
  'question': 'What is the first digit of pi?',
  'answer_real': '3'},
 {'X': 'e',
  'real_value': '2.71828',
  'question': 'What is the first digit of e?',
  'answer_real': '2'}]

In [12]:
data_level2_constants = [
    {
        "X": row["X"],
        "real_value": row["real_value"],
        "question": row["question"],
        "answer_real": row["answer_real"]
    }
    for _, row in df2_constants.iterrows()
]
data_level2_constants[:2]

[{'X': 'pi',
  'real_value': '3.14159',
  'question': 'What is pi multiplied by 3?',
  'answer_real': '9.4248'},
 {'X': 'e',
  'real_value': '2.71828',
  'question': 'What is e^2?',
  'answer_real': '7.389'}]

In [13]:
data_level3_constants = [
    {
        "X": row["X"],
        "real_value": row["real_value"],
        "question": row["question"],
        "answer_real": row["answer_real"]
    }
    for _, row in df3_constants.iterrows()
]
data_level3_constants[:2]

[{'X': 'pi',
  'real_value': '3.14159',
  'question': "What is the Earth's surface area?",
  'answer_real': '510,064,472 square kilometers'},
 {'X': 'e',
  'real_value': '2.71828',
  'question': 'If a population grows continuously at a rate of 5% per year, by what factor will it increase in 10 years?',
  'answer_real': '1.649'}]

In [16]:
data_level1_constants_mc = [
    {
        "X": row["X"],
        "real_value": row["real_value"],
        "question": row["question"],
        "answer_real": row["answer_real"],
        "Y1": row["Y1"],
        "A1": row["A1"],
        "B1": row["B1"],
        "C1": row["C1"],
        "D1": row["D1"],
        "answer1": row["answer1"],
        "answer1_real": row["answer1_real"]
    }
    for _, row in df1_constants_mc.iterrows()
]
data_level1_constants_mc[:2]

[{'X': 'pi',
  'real_value': '3.14159',
  'question': 'What is the first digit of pi?',
  'answer_real': '3',
  'Y1': '4.5',
  'A1': '2',
  'B1': '3',
  'C1': '4',
  'D1': '5',
  'answer1': 'C',
  'answer1_real': 'B'},
 {'X': 'e',
  'real_value': '2.71828',
  'question': 'What is the first digit of e?',
  'answer_real': '2',
  'Y1': '9',
  'A1': '2',
  'B1': '4',
  'C1': '7',
  'D1': '9',
  'answer1': 'D',
  'answer1_real': 'A'}]

In [17]:
data_level2_constants_mc = [
    {
        "X": row["X"],
        "real_value": row["real_value"],
        "question": row["question"],
        "answer_real": row["answer_real"],
        "Y1": row["Y1"],
        "A1": row["A1"],
        "B1": row["B1"],
        "C1": row["C1"],
        "D1": row["D1"],
        "answer1": row["answer1"],
        "answer1_real": row["answer1_real"]
    }
    for _, row in df2_constants_mc.iterrows()
]
data_level2_constants_mc[:2]

[{'X': 'pi',
  'real_value': '3.14159',
  'question': 'What is pi multiplied by 3?',
  'answer_real': '9.4248',
  'Y1': '4.5',
  'A1': '9',
  'B1': '9.4248',
  'C1': '13.5',
  'D1': '15',
  'answer1': 'C',
  'answer1_real': 'B'},
 {'X': 'e',
  'real_value': '2.71828',
  'question': 'What is e^2?',
  'answer_real': '7.389',
  'Y1': '9',
  'A1': '7.389',
  'B1': '9',
  'C1': '18',
  'D1': '81',
  'answer1': 'D',
  'answer1_real': 'A'}]

In [18]:
data_level3_constants_mc = [
    {
        "X": row["X"],
        "real_value": row["real_value"],
        "question": row["question"],
        "answer_real": row["answer_real"],
        "Y1": row["Y1"],
        "A1": row["A1"],
        "B1": row["B1"],
        "C1": row["C1"],
        "D1": row["D1"],
        "answer1": row["answer1"],
        "answer1_real": row["answer1_real"]
    }
    for _, row in df3_constants_mc.iterrows()
]
data_level3_constants_mc[:2]

[{'X': 'pi',
  'real_value': '3.14159',
  'question': "What is the Earth's surface area?",
  'answer_real': '510,064,472 square kilometers',
  'Y1': '4.5',
  'A1': '510,064,472 square kilometers',
  'B1': '730,397,538 square kilometers',
  'C1': '730,397,538 square meters',
  'D1': '510,064,472 square meters',
  'answer1': 'B',
  'answer1_real': 'A'},
 {'X': 'e',
  'real_value': '2.71828',
  'question': 'If a population grows continuously at a rate of 5% per year, by what factor will it increase in 10 years?',
  'answer_real': '1.649',
  'Y1': '9',
  'A1': '1.649',
  'B1': '9',
  'C1': '81',
  'D1': '3',
  'answer1': 'D',
  'answer1_real': 'A'}]

### **Free Form**

In [ ]:
answers1_z2 = []
answers1_cot2 = []
answers1_f2 = []

for row in data_level1_constants:
    question = row['question']
    answers1_z2.append(get_model_answer(prompt_template_z, question))
    answers1_cot2.append(get_model_answer(prompt_template_cot, question))
    answers1_f2.append(get_model_answer(prompt_template_f_constants, question))

In [ ]:
answers2_z2 = []
answers2_cot2 = []
answers2_f2 = []

for row in data_level2_constants:
    question = row['question']
    answers2_z2.append(get_model_answer(prompt_template_z, question))
    answers2_cot2.append(get_model_answer(prompt_template_cot, question))
    answers2_f2.append(get_model_answer(prompt_template_f_constants, question))

In [ ]:
answers3_z2 = []
answers3_cot2 = []
answers3_f2 = []

for row in data_level2_constants:
    question = row['question']
    answers3_z2.append(get_model_answer(prompt_template_z, question))
    answers3_cot2.append(get_model_answer(prompt_template_cot, question))
    answers3_f2.append(get_model_answer(prompt_template_f_constants, question))

In [ ]:
answers1_z_fixed2 = fix_answers(answers1_z2)
answers2_z_fixed2 = fix_answers(answers2_z2)
answers3_z_fixed2 = fix_answers(answers3_z2)

answers1_cot_fixed2 = fix_answers(answers1_cot2)
answers2_cot_fixed2 = fix_answers(answers2_cot2)
answers3_cot_fixed2 = fix_answers(answers3_cot2)

answers1_f_fixed2 = fix_answers(answers1_f2)
answers2_f_fixed2 = fix_answers(answers2_f2)
answers3_f_fixed2 = fix_answers(answers3_f2)

### **Multiple Choice**

In [ ]:
answers1_z_mc2 = []
answers1_cot_mc2 = []
answers1_f_mc2 = []

for row in data_level1_constants_mc:
    question = row['question']
    A1, B1, C1, D1 = row['A1'], row['B1'], row['C1'], row['D1']
    answers1_z_mc2.append(get_model_answer_mc(prompt_template_z_mc, question, A1, B1, C1, D1))
    answers1_cot_mc2.append(get_model_answer_mc(prompt_template_cot_mc, question, A1, B1, C1, D1))
    answers1_f_mc2.append(get_model_answer_mc(prompt_template_f_mc_constants, question, A1, B1, C1, D1))

In [ ]:
answers2_z_mc2 = []
answers2_cot_mc2 = []
answers2_f_mc2 = []

for row in data_level1_constants_mc:
    question = row['question']
    A1, B1, C1, D1 = row['A1'], row['B1'], row['C1'], row['D1']
    answers2_z_mc2.append(get_model_answer_mc(prompt_template_z_mc, question, A1, B1, C1, D1))
    answers2_cot_mc2.append(get_model_answer_mc(prompt_template_cot_mc, question, A1, B1, C1, D1))
    answers2_f_mc2.append(get_model_answer_mc(prompt_template_f_mc_constants, question, A1, B1, C1, D1))

In [ ]:
answers3_z_mc2 = []
answers3_cot_mc2 = []
answers3_f_mc2 = []

for row in data_level1_constants_mc:
    question = row['question']
    A1, B1, C1, D1 = row['A1'], row['B1'], row['C1'], row['D1']
    answers3_z_mc2.append(get_model_answer_mc(prompt_template_z_mc, question, A1, B1, C1, D1))
    answers3_cot_mc2.append(get_model_answer_mc(prompt_template_cot_mc, question, A1, B1, C1, D1))
    answers3_f_mc2.append(get_model_answer_mc(prompt_template_f_mc_constants, question, A1, B1, C1, D1))

In [ ]:
answers1_z_mc_fixed2 = fix_answers(answers1_z_mc2)
answers2_z_mc_fixed2 = fix_answers(answers2_z_mc2)
answers3_z_mc_fixed2 = fix_answers(answers3_z_mc2)

answers1_cot_mc_fixed2 = fix_answers(answers1_cot_mc2)
answers2_cot_mc_fixed2 = fix_answers(answers2_cot_mc2)
answers3_cot_mc_fixed2 = fix_answers(answers3_cot_mc2)

answers1_f_mc_fixed2 = fix_answers(answers1_f_mc2)
answers2_f_mc_fixed2 = fix_answers(answers2_f_mc2)
answers3_f_mc_fixed2 = fix_answers(answers3_f_mc2)

# **Evaluation**

In [ ]:
comparison_prompt = PromptTemplate(
    input_variables=["llm_answer", "real_answer"],
    template="""
You are tasked with comparing two answers: one provided by an LLM (the "LLM answer") and the correct answer (the "real answer"). Your job is to determine if the LLM answer matches the letter of either of the the real answer (A, B, C, or D).

**Instructions:**
1. Compare the LLM answer to the real answer carefully.
2. If the LLM answer matches the real answer, output:
   correct
3. If the LLM answer does not match the real answer, output:
   incorrect

Do not output explanations, justifications, or any additional text — only the word "correct" or "incorrect".

**LLM Answer:**  
{llm_answer}

**Real Answer:**  
{real_answer}

**Output:**
"""
)

In [ ]:
claude_model_id = "anthropic.claude-3-5-sonnet-20241022-v2:0"

def get_claude_response(messages):
    
    native_request = {
        "messages": messages,
        "max_tokens": 512,
        "temperature": 0.5,
        "anthropic_version": "bedrock-2023-05-31"
    }

    request = json.dumps(native_request)

    try:
        response = client.invoke_model(modelId=claude_model_id, body=request)
        
        model_response = json.loads(response["body"].read())

        return model_response["content"][0]["text"]
    
    except (ClientError, Exception) as e:
        print(f"ERROR: Can't invoke 'anthropic.claude-3'. Reason: {e}")
        return None

In [ ]:
def get_claude_answer(prompt_template, llm_answer, real_answer):
    claude_messages = [
        {"role": "user", "content": comparison_prompt.format(llm_answer=llm_answer, real_answer=real_answer)},
    ]
    claude_ans = get_claude_response(claude_messages)
    return claude_ans

## **Units of Measure**

### **Free Form**

In [ ]:
res1_z1 = []
res1_cot1 = []
res1_f1 = []

for i in range(len(answers1_z_fixed1)):
    llm_answer1 = answers1_z_fixed1[i]
    llm_answer2 = answers1_cot_fixed1[i]
    llm_answer3 = answers1_f_fixed1[i]
    real_answer = data_level1_units[i]["answer_real"]

    res1_z1.append(get_claude_answer(comparison_prompt, llm_answer1, real_answer))
    res1_cot1.append(get_claude_answer(comparison_prompt, llm_answer2, real_answer))
    res1_f1.append(get_claude_answer(comparison_prompt, llm_answer3, real_answer))

In [ ]:
res2_z1 = []
res2_cot1 = []
res2_f1 = []

for i in range(len(answers2_z_fixed1)):
    llm_answer1 = answers2_z_fixed1[i]
    llm_answer2 = answers2_cot_fixed1[i]
    llm_answer3 = answers2_f_fixed1[i]
    real_answer = data_level2_units[i]["answer_real"]

    res2_z1.append(get_claude_answer(comparison_prompt, llm_answer1, real_answer))
    res2_cot1.append(get_claude_answer(comparison_prompt, llm_answer2, real_answer))
    res2_f1.append(get_claude_answer(comparison_prompt, llm_answer3, real_answer))

In [ ]:
res3_z1 = []
res3_cot1 = []
res3_f1 = []

for i in range(len(answers3_z_fixed1)):
    llm_answer1 = answers3_z_fixed1[i]
    llm_answer2 = answers3_cot_fixed1[i]
    llm_answer3 = answers3_f_fixed1[i]
    real_answer = data_level3_units[i]["answer_real"]

    res3_z1.append(get_claude_answer(comparison_prompt, llm_answer1, real_answer))
    res3_cot1.append(get_claude_answer(comparison_prompt, llm_answer2, real_answer))
    res3_f1.append(get_claude_answer(comparison_prompt, llm_answer3, real_answer))

### **Multiple Choice**

In [ ]:
res1_z_mc1 = []
res1_cot_mc1 = []
res1_f_mc1 = []

for i in range(len(answers1_z_mc_fixed1)):
    llm_answer1 = answers1_z_mc_fixed1[i]
    llm_answer2 = answers1_cot_mc_fixed1[i]
    llm_answer3 = answers1_f_mc_fixed1[i]
    real_answer = data_level1_units_mc[i]["answer1_real"]

    res1_z_mc1.append(get_claude_answer(comparison_prompt, llm_answer1, real_answer))
    res1_cot_mc1.append(get_claude_answer(comparison_prompt, llm_answer2, real_answer))
    res1_f_mc1.append(get_claude_answer(comparison_prompt, llm_answer3, real_answer))

In [ ]:
res2_z_mc1 = []
res2_cot_mc1 = []
res2_f_mc1 = []

for i in range(len(answers2_z_mc_fixed1)):
    llm_answer1 = answers2_z_mc_fixed1[i]
    llm_answer2 = answers2_cot_mc_fixed1[i]
    llm_answer3 = answers2_f_mc_fixed1[i]
    real_answer = data_level2_units_mc[i]["answer1_real"]

    res2_z_mc1.append(get_claude_answer(comparison_prompt, llm_answer1, real_answer))
    res2_cot_mc1.append(get_claude_answer(comparison_prompt, llm_answer2, real_answer))
    res2_f_mc1.append(get_claude_answer(comparison_prompt, llm_answer3, real_answer))

In [ ]:
res3_z_mc1 = []
res3_cot_mc1 = []
res3_f_mc1 = []

for i in range(len(answers3_z_mc_fixed1)):
    llm_answer1 = answers3_z_mc_fixed1[i]
    llm_answer2 = answers3_cot_mc_fixed1[i]
    llm_answer3 = answers3_f_mc_fixed1[i]
    real_answer = data_level3_units_mc[i]["answer1_real"]

    res3_z_mc1.append(get_claude_answer(comparison_prompt, llm_answer1, real_answer))
    res3_cot_mc1.append(get_claude_answer(comparison_prompt, llm_answer2, real_answer))
    res3_f_mc1.append(get_claude_answer(comparison_prompt, llm_answer3, real_answer))

## **Constants**

### **Free Form**

In [ ]:
res1_z2 = []
res1_cot2 = []
res1_f2 = []

for i in range(len(answers1_z_fixed2)):
    llm_answer1 = answers1_z_fixed2[i]
    llm_answer2 = answers1_cot_fixed2[i]
    llm_answer3 = answers1_f_fixed2[i]
    real_answer = data_level1_constants[i]["answer_real"]

    res1_z2.append(get_claude_answer(comparison_prompt, llm_answer1, real_answer))
    res1_cot2.append(get_claude_answer(comparison_prompt, llm_answer2, real_answer))
    res1_f2.append(get_claude_answer(comparison_prompt, llm_answer3, real_answer))

In [ ]:
res2_z2 = []
res2_cot2 = []
res2_f2 = []

for i in range(len(answers2_z_fixed2)):
    llm_answer1 = answers2_z_fixed2[i]
    llm_answer2 = answers2_cot_fixed2[i]
    llm_answer3 = answers2_f_fixed2[i]
    real_answer = data_level2_constants[i]["answer_real"]

    res2_z2.append(get_claude_answer(comparison_prompt, llm_answer1, real_answer))
    res2_cot2.append(get_claude_answer(comparison_prompt, llm_answer2, real_answer))
    res2_f2.append(get_claude_answer(comparison_prompt, llm_answer3, real_answer))

In [ ]:
res3_z2 = []
res3_cot2 = []
res3_f2 = []

for i in range(len(answers3_z_fixed2)):
    llm_answer1 = answers3_z_fixed2[i]
    llm_answer2 = answers3_cot_fixed2[i]
    llm_answer3 = answers3_f_fixed2[i]
    real_answer = data_level3_constants[i]["answer_real"]

    res3_z2.append(get_claude_answer(comparison_prompt, llm_answer1, real_answer))
    res3_cot2.append(get_claude_answer(comparison_prompt, llm_answer2, real_answer))
    res3_f2.append(get_claude_answer(comparison_prompt, llm_answer3, real_answer))

### **Multiple Choice**

In [ ]:
res1_z_mc2 = []
res1_cot_mc2 = []
res1_f_mc2 = []

for i in range(len(answers1_z_mc_fixed2)):
    llm_answer1 = answers1_z_mc_fixed2[i]
    llm_answer2 = answers1_cot_mc_fixed2[i]
    llm_answer3 = answers1_f_mc_fixed2[i]
    real_answer = data_level1_constants_mc[i]["answer1_real"]

    res1_z_mc2.append(get_claude_answer(comparison_prompt, llm_answer1, real_answer))
    res1_cot_mc2.append(get_claude_answer(comparison_prompt, llm_answer2, real_answer))
    res1_f_mc2.append(get_claude_answer(comparison_prompt, llm_answer3, real_answer))

In [ ]:
res2_z_mc2 = []
res2_cot_mc2 = []
res2_f_mc2 = []

for i in range(len(answers2_z_mc_fixed2)):
    llm_answer1 = answers2_z_mc_fixed2[i]
    llm_answer2 = answers2_cot_mc_fixed2[i]
    llm_answer3 = answers2_f_mc_fixed2[i]
    real_answer = data_level2_constants_mc[i]["answer1_real"]

    res2_z_mc2.append(get_claude_answer(comparison_prompt, llm_answer1, real_answer))
    res2_cot_mc2.append(get_claude_answer(comparison_prompt, llm_answer2, real_answer))
    res2_f_mc2.append(get_claude_answer(comparison_prompt, llm_answer3, real_answer))

In [ ]:
res3_z_mc2 = []
res3_cot_mc2 = []
res3_f_mc2 = []

for i in range(len(answers3_z_mc_fixed2)):
    llm_answer1 = answers3_z_mc_fixed2[i]
    llm_answer2 = answers3_cot_mc_fixed2[i]
    llm_answer3 = answers3_f_mc_fixed2[i]
    real_answer = data_level3_constants_mc[i]["answer1_real"]

    res3_z_mc2.append(get_claude_answer(comparison_prompt, llm_answer1, real_answer))
    res3_cot_mc2.append(get_claude_answer(comparison_prompt, llm_answer2, real_answer))
    res3_f_mc2.append(get_claude_answer(comparison_prompt, llm_answer3, real_answer))

# Results

We present the results for evaluating the model's performance on the No Redefinition task in the tables below. The resulting tables are exported in a .txt file.

In [ ]:
from tabulate import tabulate

results_units = [
    res1_z1, res1_cot1, res1_f1, res2_z1, res2_cot1, res2_f1, res3_z1, res3_cot1, res3_f1,
    res1_z_mc1, res1_cot_mc1, res1_f_mc1, res2_z_mc1, res2_cot_mc1, res2_f_mc1, res3_z_mc1, res3_cot_mc1, res3_f_mc1
]

results_constants = [
    res1_z2, res1_cot2, res1_f2, res2_z2, res2_cot2, res2_f2, res3_z2, res3_cot2, res3_f2,
    res1_z_mc2, res1_cot_mc2, res1_f_mc2, res2_z_mc2, res2_cot_mc2, res2_f_mc2, res3_z_mc2, res3_cot_mc2, res3_f_mc2
]

def calculate_accuracy(result_list):
    correct_count = sum(1 for item in result_list if item == "correct")
    return round(100 * correct_count / len(result_list), 2)

levels = ['Level 1', 'Level 2', 'Level 3']
columns = [' z ', 'cot', 'f', 'z_mc', 'cot_mc', 'f_mc']


data_units = [
    [calculate_accuracy(res1_z1), calculate_accuracy(res1_cot1), calculate_accuracy(res1_f1), calculate_accuracy(res1_z_mc1), calculate_accuracy(res1_cot_mc1), calculate_accuracy(res1_f_mc1)],
    [calculate_accuracy(res2_z1), calculate_accuracy(res2_cot1), calculate_accuracy(res2_f1), calculate_accuracy(res2_z_mc1), calculate_accuracy(res2_cot_mc1), calculate_accuracy(res2_f_mc1)],
    [calculate_accuracy(res3_z1), calculate_accuracy(res3_cot1), calculate_accuracy(res3_f1), calculate_accuracy(res3_z_mc1), calculate_accuracy(res3_cot_mc1), calculate_accuracy(res3_f_mc1)]
]

data_constants = [
    [calculate_accuracy(res1_z2), calculate_accuracy(res1_cot2), calculate_accuracy(res1_f2), calculate_accuracy(res1_z_mc2), calculate_accuracy(res1_cot_mc2), calculate_accuracy(res1_f_mc2)],
    [calculate_accuracy(res2_z2), calculate_accuracy(res2_cot2), calculate_accuracy(res2_f2), calculate_accuracy(res2_z_mc2), calculate_accuracy(res2_cot_mc2), calculate_accuracy(res2_f_mc2)],
    [calculate_accuracy(res3_z2), calculate_accuracy(res3_cot2), calculate_accuracy(res3_f2), calculate_accuracy(res3_z_mc2), calculate_accuracy(res3_cot_mc2), calculate_accuracy(res3_f_mc2)]
]

df_constants = pd.DataFrame(data_constants, columns=columns, index=levels)
df_units = pd.DataFrame(data_units, columns=columns, index=levels)

print("Units:")
print(tabulate(df_units, headers='keys', tablefmt='grid'))
print("\nConstants:")
print(tabulate(df_constants, headers='keys', tablefmt='grid'))

In [ ]:
with open("NR_results_tables.txt", "w") as file:
    file.write("Units:\n")
    file.write(tabulate(df_units, headers='keys', tablefmt='grid'))
    file.write("\n\Constants:\n")
    file.write(tabulate(df_constants, headers='keys', tablefmt='grid'))

print("Accuracy results saved to NR_results_tables.txt")